# Collins 2026Feb beamtime local curation workflow

Use the already-ingested local catalog/zarr view (no NAS dependency), inspect relevant scans, apply per-scan sample/tag updates, verify theta/energy ranges, and correct per-scan classification (`fixed_energy` vs `fixed_angle`).

This notebook is structured as executable validation steps so each curation action is immediately checked.


In [17]:
from pathlib import Path

import matplotlib.pyplot as plt
from pyref.io import (
    apply_scan_overrides,
    get_image,
    read_beamtime_local,
    set_beamtime_scan_types,
    summarize_beamtime_scans,
)

In [7]:
BEAMTIME = Path("/Volumes/DATA/Collins/2026Feb")

assert BEAMTIME.is_absolute(), f"beamtime must be absolute: {BEAMTIME}"
print(f"beamtime exists on filesystem: {BEAMTIME.exists()}")
BEAMTIME

beamtime exists on filesystem: False


PosixPath('/Volumes/DATA/Collins/2026Feb')

In [8]:
view = read_beamtime_local(BEAMTIME, catalog_path=CATALOG_PATH, require_indexed=True)

print(f"catalog: {view.catalog_path}")
print(f"beamtime: {view.beamtime_path}")
print(f"frames: {view.frames.height}")
print(f"samples: {len(view.entries.samples)}")
print(f"scans: {len(view.entries.scans)}")

view.frames.head(5)

catalog: /Users/hduva/Library/Application Support/pyref/catalog.db
beamtime: /Volumes/DATA/Collins/2026Feb
frames: 13546
samples: 9
scans: 33


file_path,data_offset,naxis1,naxis2,bitpix,bzero,data_size,file_name,sample_name,tag,scan_number,frame_number,DATE,Beamline Energy,Sample Theta,CCD Theta,Higher Order Suppressor,EPU Polarization,EXPOSURE,Sample Name,Scan ID,Lambda,Q,beam_row,beam_col,beam_sigma,reflectivity_profile_index,reflectivity_scan_type,catalog_scan_type
str,i64,i64,i64,i64,i64,i64,str,str,str,i64,i64,str,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,i64,i64,f64,i64,str,str
"""/Volumes/DATA/Collins/2026Feb/…",20160,1000,400,16,32768,800000,"""opv_ternary_88147-00001.fits""","""opv""","""ternary""",88147,1,"""2026-02-10T14:37:17""",249.998516,0.0,0.001,null,190.0,0.0,"""opv""",88147.0,null,null,null,null,null,null,null,"""fixed_energy"""
"""/Volumes/DATA/Collins/2026Feb/…",20160,1000,400,16,32768,800000,"""opv_ternary_88147-00002.fits""","""opv""","""ternary""",88147,2,"""2026-02-10T14:37:20""",250.000403,0.0,0.0,null,190.0,0.0,"""opv""",88147.0,null,null,null,null,null,null,null,"""fixed_energy"""
"""/Volumes/DATA/Collins/2026Feb/…",20160,1000,400,16,32768,800000,"""opv_ternary_88147-00003.fits""","""opv""","""ternary""",88147,3,"""2026-02-10T14:37:23""",250.000403,0.0,0.0,null,190.0,0.0,"""opv""",88147.0,null,null,null,null,null,null,null,"""fixed_energy"""
"""/Volumes/DATA/Collins/2026Feb/…",20160,1000,400,16,32768,800000,"""opv_ternary_88147-00004.fits""","""opv""","""ternary""",88147,4,"""2026-02-10T14:37:26""",250.000403,0.0,0.0,null,190.0,0.0,"""opv""",88147.0,null,null,null,null,null,null,null,"""fixed_energy"""
"""/Volumes/DATA/Collins/2026Feb/…",20160,1000,400,16,32768,800000,"""opv_ternary_88147-00005.fits""","""opv""","""ternary""",88147,5,"""2026-02-10T14:37:29""",250.000403,0.0,0.0,null,190.0,0.0,"""opv""",88147.0,null,null,null,null,null,null,null,"""fixed_energy"""


In [9]:
summary_all = summarize_beamtime_scans(BEAMTIME, catalog_path=CATALOG_PATH)

assert summary_all.height > 0, "no scans found for beamtime"
summary_all.head(20)

scan_number,n_frames,sample_names,tags,energy_min,energy_max,theta_min,theta_max,inferred_scan_type,catalog_scan_type
i64,i64,list[str],list[str],f64,f64,f64,f64,str,str
88147,616,"[""opv""]","[""ternary""]",249.996629,284.411644,0.0,60.0,"""fixed_energy""","""fixed_energy"""
88148,724,"[""opv""]","[""ternary""]",284.39697,286.012842,0.0,60.0,"""fixed_energy""","""fixed_energy"""
88149,336,"[""opv""]","[""ternary""]",285.995529,286.005421,0.0,60.0,"""fixed_energy""","""fixed_energy"""
88150,24,"[""opv""]","[""ternary""]",249.998516,250.000403,0.0,2.182,"""fixed_energy""","""fixed_energy"""
88151,1351,"[""opv""]","[""binary""]",249.996629,286.007895,0.0,60.0,"""fixed_energy""","""fixed_energy"""
…,…,…,…,…,…,…,…,…,…
88162,456,"[""znpc_izero""]",[],249.998516,364.977313,0.0,0.0,"""fixed_angle""","""fixed_energy"""
88163,23,"[""znpc_izero""]",[],249.998516,272.009097,0.0,0.0,"""fixed_angle""","""fixed_energy"""
88164,22,"[""znpc_izero""]",[],249.998516,271.001602,20.0,20.0,"""fixed_angle""","""fixed_energy"""


In [20]:
frames_subset = view.frames.filter(view.frames["scan_number"].is_in(selected_scans))
assert frames_subset.height > 0, "no frame rows for selected scans"

img = get_image(frames_subset, 1)

plt.figure(figsize=(6, 4))
plt.imshow(img, cmap="magma")
plt.colorbar(label="counts")
plt.title(f"Corrected image preview for scan {selected_scans[0]}")
plt.tight_layout()
plt.show()

dataframe filtered


RuntimeError: [FitsError] kind=Io retryable=Temporary message=raw_pixels open source=No such file or directory (os error 2)